In [1]:
import numpy as np
from src.environment.action_space import apply_constraints


In [2]:
def apply_constraints_fix(trade_amounts: np.ndarray,
                      cash: float,
                      holdings: np.ndarray,
                      prices: np.ndarray,
                      fee_rate: float) -> np.ndarray:
    """
    Dieu chinh trade_amounts de:
        - Khong ban qua so dang giu
        - Khong mua qua so tien mat
        - Khong tao cash am
    """

    trade_amounts = trade_amounts.copy()
    n_stocks = len(trade_amounts)

    # ===== 1️ SELL constraint =====
    for i in range(n_stocks):
        if trade_amounts[i] < 0:
            max_sell = holdings[i]
            trade_amounts[i] = -min(abs(trade_amounts[i]), max_sell)

    # Tinh tien thu duoc tu ban (tru phi) de tinh vao ngan sach mua
    sell_proceeds = 0.0
    for i in range(n_stocks):
        if trade_amounts[i] < 0:
            value = abs(trade_amounts[i]) * prices[i]
            sell_proceeds += value - value * fee_rate

    # ===== BUY constraint =====
    available_cash = cash + sell_proceeds

    for i in range(n_stocks):
        if trade_amounts[i] > 0:

            shares = trade_amounts[i]
            value = shares * prices[i]
            fee = value * fee_rate
            total_cost = value + fee

            if total_cost > available_cash:
                # Tinh so co phieu toi da co the mua
                max_affordable = int(
                    available_cash / (prices[i] * (1 + fee_rate))
                )
                trade_amounts[i] = max(0, max_affordable)

                # cap nhat lai gia tri
                shares = trade_amounts[i]
                value = shares * prices[i]
                fee = value * fee_rate
                total_cost = value + fee

            available_cash -= total_cost

    return trade_amounts

In [3]:
tickers = ["ACB", "FPT", "SHB"]
prices = np.array([25_000.0, 100_000.0, 12_000.0])
fee_rate = 0.0015


def tao_kich_ban():
    """Tạo danh sách các kịch bản để chạy thử."""
    scenarios = []

    # Trường hợp 1: Có tiền mặt, không bán
    scenarios.append({
        "ten": "Trường hợp 1: Có tiền mặt, không bán",
        "cash": 50_000_000.0,
        "holdings": np.array([500, 200, 1000]),
        "trade_amounts": np.array([0, 80, 0], dtype=np.int32),
    })

    # Trường hợp 2: Hết tiền mặt, bán ACB để mua FPT
    scenarios.append({
        "ten": "Trường hợp 2: Hết tiền mặt, bán ACB để mua FPT",
        "cash": 0.0,
        "holdings": np.array([500, 0, 1000]),
        "trade_amounts": np.array([-200, 50, 0], dtype=np.int32),
    })

    # Trường hợp 3: Ít tiền mặt, bán SHB để mua FPT
    scenarios.append({
        "ten": "Trường hợp 3: Ít tiền mặt, bán SHB để mua FPT",
        "cash": 1_000_000.0,
        "holdings": np.array([0, 0, 300]),
        "trade_amounts": np.array([0, 80, -300], dtype=np.int32),
    })

    # Trường hợp 4: Bán nhiều mã để mua 1 mã
    scenarios.append({
        "ten": "Trường hợp 4: Bán nhiều mã để mua 1 mã",
        "cash": 0.0,
        "holdings": np.array([1000, 0, 500]),
        "trade_amounts": np.array([-500, 100, -500], dtype=np.int32),
    })

    return scenarios


def chay_kich_ban(constraints_fn, ten_phien_ban):
    print("=" * 70)
    print(ten_phien_ban)
    print("=" * 70)

    scenarios = tao_kich_ban()

    for sc in scenarios:
        print(f"\n--- {sc['ten']} ---")
        cash = sc["cash"]
        holdings = sc["holdings"]
        trade_amounts = sc["trade_amounts"]

        kq = constraints_fn(trade_amounts, cash, holdings, prices, fee_rate)

        print(f"Tiền mặt:     {cash:>15,.0f}")
        print(f"Holdings:     {holdings}")
        print(f"Lệnh gốc:     {trade_amounts}")
        print(f"Kết quả:      {kq}")


In [5]:
# Chay lai vơi phien ban fix
chay_kich_ban(apply_constraints_fix, "Phiên bản cũ")


Phiên bản cũ

--- Trường hợp 1: Có tiền mặt, không bán ---
Tiền mặt:          50,000,000
Holdings:     [ 500  200 1000]
Lệnh gốc:     [ 0 80  0]
Kết quả:      [ 0 80  0]

--- Trường hợp 2: Hết tiền mặt, bán ACB để mua FPT ---
Tiền mặt:                   0
Holdings:     [ 500    0 1000]
Lệnh gốc:     [-200   50    0]
Kết quả:      [-200   49    0]

--- Trường hợp 3: Ít tiền mặt, bán SHB để mua FPT ---
Tiền mặt:           1,000,000
Holdings:     [  0   0 300]
Lệnh gốc:     [   0   80 -300]
Kết quả:      [   0   45 -300]

--- Trường hợp 4: Bán nhiều mã để mua 1 mã ---
Tiền mặt:                   0
Holdings:     [1000    0  500]
Lệnh gốc:     [-500  100 -500]
Kết quả:      [-500  100 -500]


In [6]:
# Chạy với phiên bản gốc
chay_kich_ban(apply_constraints, "Phiên bản mới")

Phiên bản mới

--- Trường hợp 1: Có tiền mặt, không bán ---
Tiền mặt:          50,000,000
Holdings:     [ 500  200 1000]
Lệnh gốc:     [ 0 80  0]
Kết quả:      [ 0 80  0]

--- Trường hợp 2: Hết tiền mặt, bán ACB để mua FPT ---
Tiền mặt:                   0
Holdings:     [ 500    0 1000]
Lệnh gốc:     [-200   50    0]
Kết quả:      [-200    0    0]

--- Trường hợp 3: Ít tiền mặt, bán SHB để mua FPT ---
Tiền mặt:           1,000,000
Holdings:     [  0   0 300]
Lệnh gốc:     [   0   80 -300]
Kết quả:      [   0    9 -300]

--- Trường hợp 4: Bán nhiều mã để mua 1 mã ---
Tiền mặt:                   0
Holdings:     [1000    0  500]
Lệnh gốc:     [-500  100 -500]
Kết quả:      [-500    0 -500]
